# Looped Block-ELL Transformer — Scaling Law Explorer

**Goal**: Study how width, depth, and pruning density interact in Parcae-style looped transformers.

Architecture:
- **Prelude** (non-looped) → **Input norm** → **Core** (looped T times via `lax.scan` with diagonal injection) → **Coda** (non-looped)
- Diagonal injection: `h_new = decay * h + dt * e`, guaranteed stable by construction
- Poisson depth sampling per sequence, BPTT truncation, sequences frozen via `jnp.where`
- MLP pruning via CMS (gradient Frobenius norm tile scoring), masked dense weights

**Change Cell 2 to run a sweep.** Everything else is automatic.

> ⚡ Runtime: TPU v2-8 or v3-8 recommended. Go to `Runtime → Change runtime type → TPU`.

## Cell 1 — Setup & Install

Installs JAX with TPU support, Flax, Optax, and helpers. Run once per session.

In [ ]:
# ─── Install ──────────────────────────────────────────────────────────────────
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# JAX TPU wheel (pins to a stable libtpu version — update if Colab TPU changes)
_pip("jax[tpu]", "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html")
_pip("flax", "optax", "wandb", "datasets", "transformers", "tqdm")

import jax
import jax.numpy as jnp
print(f"JAX {jax.__version__}")
print(f"Devices ({len(jax.devices())}): {jax.devices()}")
assert len(jax.devices()) >= 4, "Expected TPU with ≥4 devices. Check runtime type."

## Cell 2 — Experiment Config  ✏️ SWEEP HERE

Edit these values to explore the three scaling axes:
- **Width**: `D_MODEL` / `D_FF` / `N_HEADS`
- **Depth**: `MEAN_DEPTH` / `BPTT_DEPTH`
- **Prune**: `PRUNE_FRACTION` / `TARGET_ROUNDS`

Typical sweep configs:
```python
# Width sweep (fix depth=8, no pruning)
D_MODEL = 256 | 384 | 512 | 768 | 1024

# Depth sweep (fix d=512, no pruning)
MEAN_DEPTH = 2 | 4 | 6 | 8 | 12 | 16

# Prune sweep (fix d=512, depth=8)
PRUNE_FRACTION = 0.05 | 0.10 | 0.15 | 0.20
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  EXPERIMENT CONFIG — Change these to explore scaling laws
# ═══════════════════════════════════════════════════════════════════════════════
EXPERIMENT_NAME = "width_sweep_d512"

# ── Width axis ────────────────────────────────────────────────────────────────
D_MODEL = 512        # try: 256, 384, 512, 768, 1024
N_HEADS = 8          # must divide D_MODEL evenly
D_FF    = 2048       # typically 4 * D_MODEL

# ── Layer structure ───────────────────────────────────────────────────────────
N_PRELUDE = 1        # non-looped layers before core
N_CORE    = 4        # unique layers in the looped core (shared weights)
N_CODA    = 1        # non-looped layers after core

# ── Depth axis ────────────────────────────────────────────────────────────────
MEAN_DEPTH  = 8      # try: 2, 4, 6, 8, 12, 16
MAX_DEPTH   = 32     # Poisson clamp upper bound
BPTT_DEPTH  = 4      # grad iterations (rest are stop_gradient)
USE_POISSON = True   # per-sequence variable depth (set False for fixed)

# ── Prune axis ────────────────────────────────────────────────────────────────
TILE_SIZE      = 16     # MLP tile granularity (d_model, d_ff must be divisible)
PRUNE_FRACTION = 0.10   # fraction of alive tiles pruned per round; try 0.05–0.20
PRUNE_INTERVAL = 4600   # steps between prune rounds
PRUNE_START    = 5000   # step to begin pruning (warmup dense first)
PRUNE_END      = 60000  # step to stop pruning (train sparse rest)
SCORE_INTERVAL = 10     # steps between CMS score normalisation

# ── Training ──────────────────────────────────────────────────────────────────
TOTAL_STEPS  = 50000
BATCH_SIZE   = 8
LR           = 6e-4
WARMUP_STEPS = 2000
GRAD_CLIP    = 1.0
MAX_SEQ_LEN  = 1024
VOCAB_SIZE   = 49152    # StarCoder2 tokenizer
WEIGHT_DECAY = 0.1
SEED         = 42

# ── Logging ───────────────────────────────────────────────────────────────────
WANDB_PROJECT = "looped-blockell-scaling"
EVAL_INTERVAL = 2500
LOG_INTERVAL  = 10

# ─── Derived / sanity checks ──────────────────────────────────────────────────
assert D_MODEL % N_HEADS   == 0, f"D_MODEL={D_MODEL} not divisible by N_HEADS={N_HEADS}"
assert D_MODEL % TILE_SIZE == 0, f"D_MODEL={D_MODEL} not divisible by TILE_SIZE={TILE_SIZE}"
assert D_FF    % TILE_SIZE == 0, f"D_FF={D_FF} not divisible by TILE_SIZE={TILE_SIZE}"
N_LAYERS  = N_PRELUDE + N_CORE + N_CODA

N_PRUNE_ROUNDS = max(0, (PRUNE_END - PRUNE_START) // PRUNE_INTERVAL)
FINAL_DENSITY  = (1.0 - PRUNE_FRACTION) ** N_PRUNE_ROUNDS if N_PRUNE_ROUNDS > 0 else 1.0
EFFECTIVE_DEPTH = N_PRELUDE + N_CORE * MEAN_DEPTH + N_CODA

print(f"Model : d={D_MODEL}, {N_PRELUDE}p+{N_CORE}c×T+{N_CODA}coda")
print(f"Depth : mean_T={MEAN_DEPTH}, bptt={BPTT_DEPTH}, effective={EFFECTIVE_DEPTH}")
print(f"Prune : {N_PRUNE_ROUNDS} rounds @ {PRUNE_FRACTION:.0%}/round → "
      f"{FINAL_DENSITY:.1%} final density")
print(f"Train : {TOTAL_STEPS} steps, batch={BATCH_SIZE}, lr={LR}")

## Cell 3 — Data Loading

Downloads OpenWebText from HuggingFace and tokenises with the GPT-2 tokeniser (tiktoken).  
All tokens are pre-packed into a flat buffer; batches are sampled randomly from it.  
Eval uses a fixed held-out slice.  

> First run takes ~5–10 min to download and tokenise. Subsequent runs use the Colab cache.

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm

# ─── Tokeniser ────────────────────────────────────────────────────────────────
print("Loading StarCoder2 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-7b", trust_remote_code=True)
EOT = tokenizer.eos_token_id

# ─── Download & tokenise ──────────────────────────────────────────────────────
TRAIN_TOKENS  = 200_000_000   # 200M tokens in train buffer
VAL_TOKENS    = 10_000_000    # 10M tokens held out for eval

def build_token_buffer(target_tokens: int, skip_tokens: int = 0) -> np.ndarray:
    """Stream and tokenise Dolma until we have `target_tokens`."""
    ds = load_dataset("allenai/dolma", split="train", streaming=True, trust_remote_code=True)
    buf = []
    skipped = 0
    with tqdm(total=target_tokens, unit="tok", desc="tokenise") as pbar:
        for sample in ds:
            text = sample.get("text", "")
            if not text:
                continue
            ids = tokenizer.encode(text, add_special_tokens=False)
            ids.append(EOT)
            if skipped < skip_tokens:
                skipped += len(ids)
                continue
            buf.extend(ids)
            pbar.update(len(ids))
            if len(buf) >= target_tokens:
                break
    return np.array(buf[:target_tokens], dtype=np.int32)

print("Loading train data (Dolma v1.7)...")
train_buf = build_token_buffer(TRAIN_TOKENS)
print("Loading val data...")
val_buf = build_token_buffer(VAL_TOKENS, skip_tokens=TRAIN_TOKENS)

print(f"Train buffer: {len(train_buf):,} tokens  ({len(train_buf)/1e6:.0f}M)")
print(f"Val   buffer: {len(val_buf):,} tokens  ({len(val_buf)/1e6:.0f}M)")


# ─── Batch sampler ────────────────────────────────────────────────────────────
rng = np.random.default_rng(SEED)

def get_batch(buf: np.ndarray, batch_size: int = BATCH_SIZE, seq_len: int = MAX_SEQ_LEN):
    """Sample `batch_size` random (seq_len+1) windows and split into x/y."""
    starts = rng.integers(0, len(buf) - seq_len - 1, size=batch_size)
    seqs   = np.stack([buf[s:s + seq_len + 1].astype(np.int32) for s in starts])
    x = jnp.asarray(seqs[:, :-1])
    y = jnp.asarray(seqs[:, 1:])
    return x, y

# smoke-test
x, y = get_batch(train_buf)
print(f"Batch shape: x={x.shape}, y={y.shape}, dtype={x.dtype}")


## Cell 4 — Model Implementation

All model code in one cell. Key components:

- **RMSNorm** — no bias, fp32 accumulation
- **RoPE** — precomputed frequency table, applied per QK head
- **MultiHeadAttention** — causal, no bias projections
- **MLPBlock** — dense weights with an external tile mask (mask applied via reshape-multiply)
- **DiagonalInjection** — `decay = exp(softplus(log_dt) * -exp(log_A))`, guaranteed ∈ (0,1)
- **TransformerBlock** — pre-norm residual
- **LoopedTransformer** — `lax.scan` over core blocks, `jax.checkpoint` for O(1) activation memory

**MLP pruning strategy**: weights stay dense, a boolean tile mask `[R, C]` (where R=d_out//B, C=d_in//B)
is applied by zeroing tile-shaped regions before the matmul. This is simpler than full Block-ELL
sparse indexing and produces identical scaling-law results since the gradient structure is the same.

In [ ]:
import math
from functools import partial
from typing import Optional, NamedTuple
from dataclasses import dataclass

import jax
import jax.numpy as jnp
import flax.linen as nn
import optax


# ─── RMSNorm ──────────────────────────────────────────────────────────────────

class RMSNorm(nn.Module):
    eps: float = 1e-6

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        scale = self.param("scale", nn.initializers.ones, (x.shape[-1],))
        # Accumulate in fp32 for numerical stability
        x32 = x.astype(jnp.float32)
        rms  = jnp.sqrt((x32 ** 2).mean(axis=-1, keepdims=True) + self.eps)
        return ((x32 / rms) * scale).astype(x.dtype)


# ─── RoPE ─────────────────────────────────────────────────────────────────────

def precompute_rope_freqs(head_dim: int, max_seq_len: int, theta: float = 10000.0) -> jnp.ndarray:
    """Return [max_seq_len, head_dim//2, 2] of (cos, sin) pairs."""
    half     = head_dim // 2
    inv_freq = 1.0 / (theta ** (jnp.arange(half, dtype=jnp.float32) / half))
    t        = jnp.arange(max_seq_len, dtype=jnp.float32)
    freqs    = jnp.outer(t, inv_freq)   # [T, half]
    return jnp.stack([jnp.cos(freqs), jnp.sin(freqs)], axis=-1)  # [T, half, 2]


def apply_rope(x: jnp.ndarray, freqs: jnp.ndarray) -> jnp.ndarray:
    """Rotate x [B, H, S, head_dim] with freqs [S, head_dim//2, 2]."""
    B, H, S, D = x.shape
    half = D // 2
    x1, x2 = x[..., :half], x[..., half:]
    cos = freqs[:S, :, 0][None, None]   # [1, 1, S, half]
    sin = freqs[:S, :, 1][None, None]
    return jnp.concatenate([x1 * cos - x2 * sin, x1 * sin + x2 * cos], axis=-1)


# ─── MLP tile mask helpers ────────────────────────────────────────────────────

def apply_tile_mask(weight: jnp.ndarray, mask: jnp.ndarray, tile_size: int = 16) -> jnp.ndarray:
    """Zero out pruned tiles in a dense weight [out, in].
    
    mask : [R, C] bool, where R=out//tile_size, C=in//tile_size.
    Tiles where mask=False are set to zero.
    """
    out_f, in_f = weight.shape
    R = out_f // tile_size
    C = in_f  // tile_size
    # Expand mask from [R, C] to [out_f, in_f] via repeat
    mask_expanded = jnp.repeat(jnp.repeat(mask, tile_size, axis=0), tile_size, axis=1)  # [out_f, in_f]
    return weight * mask_expanded.astype(weight.dtype)


def compute_tile_scores(grad_weight: jnp.ndarray, tile_size: int = 16) -> jnp.ndarray:
    """Per-tile Frobenius norm of gradient [out, in] → scores [R, C]."""
    out_f, in_f = grad_weight.shape
    R = out_f // tile_size
    C = in_f  // tile_size
    # Reshape into tiles: [R, tile, C, tile]
    blocked = grad_weight.reshape(R, tile_size, C, tile_size)
    return jnp.sqrt((blocked.astype(jnp.float32) ** 2).sum(axis=(1, 3)))   # [R, C]


# ─── Multi-head causal attention ──────────────────────────────────────────────

class MultiHeadAttention(nn.Module):
    n_heads:     int
    d_model:     int
    max_seq_len: int = 1024

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        B, S, D = x.shape
        head_dim = self.d_model // self.n_heads

        # Fused QKV projection
        qkv = nn.Dense(3 * self.d_model, use_bias=False, dtype=jnp.bfloat16,
                       name="qkv")(x)          # [B, S, 3D]
        q, k, v = jnp.split(qkv, 3, axis=-1)  # each [B, S, D]

        def reshape(t):
            return t.reshape(B, S, self.n_heads, head_dim).transpose(0, 2, 1, 3)

        q, k, v = reshape(q), reshape(k), reshape(v)   # [B, H, S, head_dim]

        # RoPE (fp32 for precision, cast back after)
        freqs = precompute_rope_freqs(head_dim, self.max_seq_len)
        q = apply_rope(q.astype(jnp.float32), freqs).astype(jnp.bfloat16)
        k = apply_rope(k.astype(jnp.float32), freqs).astype(jnp.bfloat16)

        # Causal scaled dot-product attention
        scale       = math.sqrt(head_dim)
        attn_logits = jnp.matmul(q, k.transpose(0, 1, 3, 2)) / scale  # [B, H, S, S]
        causal_mask = jnp.tril(jnp.ones((S, S), dtype=jnp.bool_))[None, None]
        NEG_INF     = jnp.finfo(jnp.bfloat16).min
        attn_logits = jnp.where(causal_mask, attn_logits, NEG_INF)
        attn_w      = jax.nn.softmax(attn_logits.astype(jnp.float32), axis=-1).astype(jnp.bfloat16)

        out = jnp.matmul(attn_w, v).transpose(0, 2, 1, 3).reshape(B, S, D)  # [B, S, D]
        return nn.Dense(self.d_model, use_bias=False, dtype=jnp.bfloat16,
                        name="out")(out)


# ─── MLP block (dense weights + external tile mask) ───────────────────────────

class MLPBlock(nn.Module):
    """Two-layer GELU MLP. fc1/fc2 weights stay dense; tile mask is applied externally.

    The caller passes `fc1_mask` and `fc2_mask` ([R, C] bool arrays) to zero pruned tiles.
    When masks are None, operates as a fully dense MLP.
    """
    d_model:   int
    d_ff:      int
    tile_size: int = 16

    @nn.compact
    def __call__(
        self,
        x: jnp.ndarray,
        fc1_mask: Optional[jnp.ndarray] = None,
        fc2_mask: Optional[jnp.ndarray] = None,
    ) -> jnp.ndarray:
        # fc1: d_model → d_ff
        w1 = self.param("fc1_w", nn.initializers.normal(0.02), (self.d_ff, self.d_model))
        b1 = self.param("fc1_b", nn.initializers.zeros, (self.d_ff,))
        # fc2: d_ff → d_model
        w2 = self.param("fc2_w", nn.initializers.normal(0.02 / math.sqrt(2)), (self.d_model, self.d_ff))
        b2 = self.param("fc2_b", nn.initializers.zeros, (self.d_model,))

        w1_eff = apply_tile_mask(w1, fc1_mask, self.tile_size) if fc1_mask is not None else w1
        w2_eff = apply_tile_mask(w2, fc2_mask, self.tile_size) if fc2_mask is not None else w2

        # Cast to bf16 for TPU MXU efficiency
        x16   = x.astype(jnp.bfloat16)
        w1_16 = w1_eff.astype(jnp.bfloat16)
        w2_16 = w2_eff.astype(jnp.bfloat16)

        hidden = jnp.dot(x16, w1_16.T) + b1.astype(jnp.bfloat16)   # [B, S, d_ff]
        hidden = jax.nn.gelu(hidden)
        out    = jnp.dot(hidden, w2_16.T) + b2.astype(jnp.bfloat16) # [B, S, d_model]
        return out


# ─── Transformer block (pre-norm) ─────────────────────────────────────────────

class TransformerBlock(nn.Module):
    d_model:     int
    n_heads:     int
    d_ff:        int
    max_seq_len: int  = 1024
    tile_size:   int  = 16

    @nn.compact
    def __call__(
        self,
        x: jnp.ndarray,
        fc1_mask: Optional[jnp.ndarray] = None,
        fc2_mask: Optional[jnp.ndarray] = None,
    ) -> jnp.ndarray:
        x = x + MultiHeadAttention(
            n_heads=self.n_heads,
            d_model=self.d_model,
            max_seq_len=self.max_seq_len,
            name="attn",
        )(RMSNorm(name="norm_attn")(x))
        x = x + MLPBlock(
            d_model=self.d_model,
            d_ff=self.d_ff,
            tile_size=self.tile_size,
            name="mlp",
        )(RMSNorm(name="norm_mlp")(x), fc1_mask, fc2_mask)
        return x


# ─── Diagonal injection ───────────────────────────────────────────────────────

class DiagonalInjection(nn.Module):
    """SSM-style state gate: decay = exp(softplus(log_dt) * -exp(log_A)) ∈ (0,1).

    h_new = decay * h + dt * e

    Stability is guaranteed by construction: decay is always in (0, 1).
    Gradients flow through log_A and log_dt in the grad (k_max) iterations.
    """
    d_model:    int
    init_decay: float = 0.447   # sqrt(1/5), Parcae default

    @nn.compact
    def __call__(self, h: jnp.ndarray, e: jnp.ndarray) -> jnp.ndarray:
        # log_A: forced negative → A = -exp(log_A) < 0
        log_A = self.param("log_A", nn.initializers.zeros, (self.d_model,))

        # log_dt initialised so that softplus(log_dt) = -log(init_decay)
        target_dt      = -math.log(self.init_decay)             # ≈ 0.806
        init_log_dt    = math.log(math.exp(target_dt) - 1.0)   # ≈ 0.533
        log_dt = self.param("log_dt",
                            nn.initializers.constant(init_log_dt),
                            (self.d_model,))

        A     = -jnp.exp(log_A)         # (−∞, 0)
        dt    = jax.nn.softplus(log_dt) # (0, +∞)
        decay = jnp.exp(dt * A)         # (0, 1)
        return decay * h + dt * e


# ─── Looped Transformer ───────────────────────────────────────────────────────

class LoopedTransformer(nn.Module):
    """Parcae-style looped transformer.

    Forward receives tile masks for all blocks packed into a pytree:
        masks: dict keyed by block path → {'fc1': [R,C] bool, 'fc2': [R,C] bool}
    None masks = fully dense (no pruning).

    n_max, k_max MUST be Python ints (resolved outside jit) so lax.scan
    gets a static iteration count.
    """
    d_model:     int
    n_heads:     int
    d_ff:        int
    n_prelude:   int
    n_core:      int
    n_coda:      int
    vocab_size:  int
    max_seq_len: int = 1024
    tile_size:   int = 16

    @nn.compact
    def __call__(
        self,
        input_ids:  jnp.ndarray,           # [B, S]
        depths:     jnp.ndarray,           # [B] per-sequence loop count
        n_max:      int,                   # Python int — no-grad iterations
        k_max:      int,                   # Python int — grad iterations
        labels:     Optional[jnp.ndarray] = None,  # [B, S]
        masks:      Optional[dict]         = None,  # pruning masks
    ) -> dict:
        cfg = dict(
            d_model=self.d_model, n_heads=self.n_heads, d_ff=self.d_ff,
            max_seq_len=self.max_seq_len, tile_size=self.tile_size,
        )
        total_iters: int = n_max + k_max   # static Python int for scan
        B, S = input_ids.shape

        def _get_masks(name):
            if masks is None:
                return None, None
            entry = masks.get(name, {})
            return entry.get("fc1"), entry.get("fc2")

        # ── 1. Embedding (weight-tied with lm_head) ──────────────────────────
        embed = nn.Embed(
            num_embeddings=self.vocab_size,
            features=self.d_model,
            embedding_init=nn.initializers.normal(0.02),
            name="embed",
        )
        x = embed(input_ids).astype(jnp.bfloat16)  # [B, S, d_model]

        # ── 2. Prelude ───────────────────────────────────────────────────────
        for i in range(self.n_prelude):
            m1, m2 = _get_masks(f"prelude_{i}")
            x = TransformerBlock(**cfg, name=f"prelude_{i}")(x, m1, m2)

        # ── 3. Input norm → inject signal e ──────────────────────────────────
        e = RMSNorm(name="input_norm")(x)   # [B, S, d_model]
        h = jnp.zeros_like(e)              # hidden state initialised to 0

        # ── 4. Build core blocks (shared weights — closed over by scan) ───────
        injection   = DiagonalInjection(d_model=self.d_model, name="injection")
        core_blocks = [TransformerBlock(**cfg, name=f"core_{i}") for i in range(self.n_core)]

        # Gather core masks once (outside scan body, same value every iteration)
        core_masks = [_get_masks(f"core_{i}") for i in range(self.n_core)]

        # ── 5. lax.scan core loop ─────────────────────────────────────────────
        def scan_body(h_carry, step_idx):
            h_new = injection(h_carry, e)
            for blk, (m1, m2) in zip(core_blocks, core_masks):
                h_new = blk(h_new, m1, m2)

            # No-grad for the first n_max steps
            is_nograd = step_idx < n_max
            h_new = jnp.where(is_nograd, jax.lax.stop_gradient(h_new), h_new)

            # Freeze finished sequences via jnp.where (NOT zeroing)
            active = (step_idx < depths)[:, None, None]   # [B, 1, 1]
            h_next = jnp.where(active, h_new, h_carry)
            return h_next, None

        scan_body_ckpt = jax.checkpoint(scan_body, prevent_cse=False)

        if total_iters > 0:
            h, _ = jax.lax.scan(
                scan_body_ckpt,
                h,
                jnp.arange(total_iters, dtype=jnp.int32),
            )

        # ── 6. Coda ───────────────────────────────────────────────────────────
        x = h
        for i in range(self.n_coda):
            m1, m2 = _get_masks(f"coda_{i}")
            x = TransformerBlock(**cfg, name=f"coda_{i}")(x, m1, m2)

        # ── 7. Final norm + weight-tied LM head ───────────────────────────────
        x = RMSNorm(name="final_norm")(x)
        logits = embed.attend(x)   # [B, S, vocab_size] — weight-tied

        # ── 8. Loss ───────────────────────────────────────────────────────────
        loss = None
        if labels is not None:
            loss = optax.softmax_cross_entropy_with_integer_labels(
                logits.reshape(-1, self.vocab_size),
                labels.reshape(-1),
            ).mean()

        return {"logits": logits, "loss": loss}


# ─── Model factory ────────────────────────────────────────────────────────────

def make_model() -> LoopedTransformer:
    return LoopedTransformer(
        d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF,
        n_prelude=N_PRELUDE, n_core=N_CORE, n_coda=N_CODA,
        vocab_size=VOCAB_SIZE, max_seq_len=MAX_SEQ_LEN, tile_size=TILE_SIZE,
    )


def count_params(params) -> int:
    return sum(p.size for p in jax.tree_util.tree_leaves(params))


print("Model classes defined. ✓")

## Cell 5 — CMS Scoring & Pruning

Pure-functional CMS (Continuum Memory System) implementation compatible with `jax.jit`.

**State**: per-layer `CMSState` holds accumulated gradient tile norms.

**Timing** (critical):
1. `accumulate_scores(cms, grad_w)` — called with raw gradients **before** optax update
2. `score_step(cms)` — normalise every `SCORE_INTERVAL` steps  
3. `prune_tiles(cms, mask, PRUNE_FRACTION)` — per-row bottom-k every `PRUNE_INTERVAL` steps
4. After pruning: re-zero optimizer momentum/variance for newly dead tiles

In [ ]:
from typing import Tuple


# ─── CMS state ────────────────────────────────────────────────────────────────

class CMSState(NamedTuple):
    """Per-MLP weight CMS state.  Carried as a plain pytree (NamedTuple).

    Fields
    ------
    scores      : [R, C] float32 — accumulated gradient Frobenius norms
    alive       : [R, C] bool    — True = tile is alive
    accum_count : scalar int32   — steps since last normalisation
    """
    scores:      jnp.ndarray   # [R, C]
    alive:       jnp.ndarray   # [R, C] bool
    accum_count: jnp.ndarray   # scalar


def init_cms(d_out: int, d_in: int, tile_size: int) -> CMSState:
    R = d_out // tile_size
    C = d_in  // tile_size
    return CMSState(
        scores=jnp.zeros((R, C), dtype=jnp.float32),
        alive=jnp.ones((R, C),  dtype=jnp.bool_),
        accum_count=jnp.array(0, dtype=jnp.int32),
    )


def accumulate_scores(cms: CMSState, grad_w: jnp.ndarray, tile_size: int = TILE_SIZE) -> CMSState:
    """Add per-tile gradient Frobenius norms to the score buffer.

    Call BEFORE optax apply_updates — mirroring the PyTorch contract of
    calling after loss.backward() and before optimizer.step().

    grad_w : [d_out, d_in]
    """
    tile_scores = compute_tile_scores(grad_w, tile_size)       # [R, C]
    tile_scores = jnp.where(cms.alive, tile_scores, 0.0)       # dead tiles stay 0
    return CMSState(
        scores=cms.scores + tile_scores,
        alive=cms.alive,
        accum_count=cms.accum_count + 1,
    )


def score_step(cms: CMSState) -> CMSState:
    """Normalise accumulated scores by step count. Call every SCORE_INTERVAL steps."""
    n = jnp.maximum(cms.accum_count.astype(jnp.float32), 1.0)
    return CMSState(
        scores=cms.scores / n,
        alive=cms.alive,
        accum_count=jnp.array(0, dtype=jnp.int32),
    )


def prune_tiles(cms: CMSState, fraction: float) -> CMSState:
    """Per-row: mark the bottom `fraction` of alive tiles as dead.

    Dead tiles get score 0 and alive=False. Col indices are not stored here;
    the caller reads cms.alive as the new mask for apply_tile_mask.
    """
    def _prune_row(scores_row: jnp.ndarray, alive_row: jnp.ndarray):
        C = scores_row.shape[0]
        n_alive  = alive_row.sum().astype(jnp.float32)
        n_to_kill = jnp.maximum(1, jnp.floor(n_alive * fraction).astype(jnp.int32))

        # Dead tiles masked to +inf so they're never selected for killing
        masked = jnp.where(alive_row, scores_row, jnp.inf)
        sorted_idx   = jnp.argsort(masked)               # ascending
        kill_pos     = jnp.arange(C) < n_to_kill         # first n_to_kill slots
        kill_at_orig = jnp.zeros(C, dtype=jnp.bool_).at[sorted_idx].set(kill_pos)
        kill_at_orig = kill_at_orig & alive_row           # only kill alive tiles
        new_alive    = alive_row & ~kill_at_orig
        return new_alive

    new_alive = jax.vmap(_prune_row)(cms.scores, cms.alive)  # [R, C]
    new_scores = jnp.where(new_alive, cms.scores, 0.0)
    return CMSState(scores=new_scores, alive=new_alive, accum_count=cms.accum_count)


def get_density(cms: CMSState) -> float:
    return float(cms.alive.mean())


# ─── CMS registry ─────────────────────────────────────────────────────────────
# Keys follow param path pattern: e.g. "prelude_0/mlp/fc1_w"

def _build_cms_registry(n_prelude: int, n_core: int, n_coda: int,
                         d_model: int, d_ff: int, tile_size: int) -> dict:
    """Create empty CMSState for every MLP weight in the model."""
    reg = {}
    block_names = (
        [f"prelude_{i}" for i in range(n_prelude)]
        + [f"core_{i}"   for i in range(n_core)]
        + [f"coda_{i}"   for i in range(n_coda)]
    )
    for name in block_names:
        # fc1: d_ff × d_model,  fc2: d_model × d_ff
        reg[f"{name}/mlp/fc1_w"] = init_cms(d_ff,    d_model, tile_size)
        reg[f"{name}/mlp/fc2_w"] = init_cms(d_model, d_ff,    tile_size)
    return reg


def cms_to_masks(cms_reg: dict) -> dict:
    """Convert CMS registry to the masks dict expected by LoopedTransformer.

    Returns: {block_name: {'fc1': [R,C] bool, 'fc2': [R,C] bool}}
    """
    masks = {}
    for key, state in cms_reg.items():
        # key = "prelude_0/mlp/fc1_w"
        parts    = key.split("/")
        blk_name = parts[0]               # e.g. "prelude_0"
        fc_key   = parts[-1].replace("_w", "")  # fc1 or fc2
        masks.setdefault(blk_name, {})[fc_key] = state.alive
    return masks


print("CMS scoring & pruning defined. ✓")

## Cell 6 — Training Loop

Three phases logged separately in wandb:
- **DENSE** — full dense MLP (no masks)
- **PRUNE R{n}** — active pruning rounds
- **SPARSE** — training continues at final density after PRUNE_END

CMS accumulation happens after `jax.grad()` and before `optax.apply_updates()`.  
Depth is sampled in Python (host) before each `jit`-ted train step.

In [ ]:
import time
import wandb

# ─── Depth sampling (outside jit) ────────────────────────────────────────────

def sample_depth_plan(key: jax.Array, batch_size: int, mean_depth: int,
                      max_depth: int, bptt_depth: int, use_poisson: bool):
    """Returns (depths [B] jnp.int32, n_max Python int, k_max Python int).

    Must be called OUTSIDE jit — int(jnp.max()) triggers a host sync.
    """
    if use_poisson:
        total = jax.random.poisson(key, lam=mean_depth, shape=(batch_size,))
        total = jnp.clip(total, 1, max_depth).astype(jnp.int32)
    else:
        total = jnp.full((batch_size,), mean_depth, dtype=jnp.int32)

    k_per_seq = jnp.minimum(total, bptt_depth)
    n_per_seq = total - k_per_seq

    n_max = int(n_per_seq.max())
    k_max = int(k_per_seq.max())
    return total, n_max, k_max


# ─── Optimizer ────────────────────────────────────────────────────────────────

def make_optimizer():
    warmup  = optax.linear_schedule(0.0, LR, WARMUP_STEPS)
    # 1M-horizon cosine (never decays to 0 within our run)
    decay   = optax.cosine_decay_schedule(LR, decay_steps=1_000_000, alpha=0.1)
    sched   = optax.join_schedules([warmup, decay], [WARMUP_STEPS])
    return optax.chain(
        optax.clip_by_global_norm(GRAD_CLIP),
        optax.adamw(sched, weight_decay=WEIGHT_DECAY, b1=0.9, b2=0.95),
    )


# ─── Opt state mask zeroing (after pruning) ───────────────────────────────────

def zero_opt_state_for_dead_tiles(
    opt_state, params, cms_reg: dict, tile_size: int = TILE_SIZE
):
    """Multiply adamw mu/nu for fc1_w/fc2_w by the alive mask.

    This prevents momentum from carrying dead tile updates forward.
    We identify the ScaleByAdamState (mu, nu) inside the optax chain and
    apply the tile mask to the corresponding weight slots.
    """
    # Build a leaves-to-mask map keyed by param path
    masks_flat = {}  # param path → expanded bool mask [out, in]
    for cms_key, state in cms_reg.items():
        parts  = cms_key.split("/")
        blk    = parts[0]   # e.g. "core_0"
        fc     = parts[-1]  # e.g. "fc1_w"

        # Reconstruct full [out, in] bool from tile alive mask
        R, C   = state.alive.shape
        m_exp  = jnp.repeat(jnp.repeat(state.alive, tile_size, axis=0), tile_size, axis=1)
        masks_flat[(blk, fc)] = m_exp.astype(jnp.float32)

    def _apply_mask_to_state(path, leaf):
        """path is a tuple of keys/strings from the pytree traversal."""
        # Extract block and fc name from path
        path_str = "/".join(str(p) for p in path)
        for (blk, fc), mask in masks_flat.items():
            if blk in path_str and fc in path_str:
                if leaf.shape == mask.shape:
                    return leaf * mask
        return leaf

    # Traverse optax state and apply masks where shape matches
    def _mask_tree(tree):
        leaves, treedef = jax.tree_util.tree_flatten_with_path(tree)
        new_leaves = []
        for path, leaf in leaves:
            if hasattr(leaf, 'shape') and len(leaf.shape) == 2:
                path_str = "/".join(str(p) for p in path)
                for (blk, fc), mask in masks_flat.items():
                    if blk in path_str and fc in path_str and leaf.shape == mask.shape:
                        leaf = leaf * mask
                        break
            new_leaves.append(leaf)
        return jax.tree_util.tree_unflatten(treedef, new_leaves)

    return _mask_tree(opt_state)


# ─── Phase label ──────────────────────────────────────────────────────────────

def phase_label(step: int, prune_round: int) -> str:
    if step < PRUNE_START or N_PRUNE_ROUNDS == 0:
        return "DENSE"
    if PRUNE_START <= step < PRUNE_END:
        return f"PRUNE R{prune_round}"
    return "SPARSE"


# ─── Main training loop ───────────────────────────────────────────────────────

def train():
    key    = jax.random.PRNGKey(SEED)
    key, k_init = jax.random.split(key)

    model  = make_model()
    tx     = make_optimizer()

    # ── Init params ───────────────────────────────────────────────────────────
    dummy_ids    = jnp.ones((BATCH_SIZE, MAX_SEQ_LEN), dtype=jnp.int32)
    dummy_depths = jnp.full((BATCH_SIZE,), MEAN_DEPTH, dtype=jnp.int32)
    n_max0 = max(0, MEAN_DEPTH - BPTT_DEPTH)
    k_max0 = min(MEAN_DEPTH, BPTT_DEPTH)

    params    = model.init(k_init, dummy_ids, dummy_depths, n_max0, k_max0)["params"]
    opt_state = tx.init(params)
    n_params  = count_params(params)
    print(f"Parameters: {n_params:,} ({n_params/1e6:.1f}M)")

    # ── CMS registry (one per MLP weight) ─────────────────────────────────────
    cms_reg = _build_cms_registry(
        N_PRELUDE, N_CORE, N_CODA, D_MODEL, D_FF, TILE_SIZE
    )
    current_masks = None   # None = fully dense
    prune_round   = 0

    # ── wandb ─────────────────────────────────────────────────────────────────
    wandb.init(
        project=WANDB_PROJECT,
        name=EXPERIMENT_NAME,
        config={
            "d_model": D_MODEL, "n_heads": N_HEADS, "d_ff": D_FF,
            "n_prelude": N_PRELUDE, "n_core": N_CORE, "n_coda": N_CODA,
            "mean_depth": MEAN_DEPTH, "bptt_depth": BPTT_DEPTH,
            "effective_depth": EFFECTIVE_DEPTH,
            "n_params": n_params,
            "lr": LR, "total_steps": TOTAL_STEPS, "batch_size": BATCH_SIZE,
            "prune_fraction": PRUNE_FRACTION, "prune_interval": PRUNE_INTERVAL,
            "prune_start": PRUNE_START, "prune_end": PRUNE_END,
            "n_prune_rounds": N_PRUNE_ROUNDS, "final_density": FINAL_DENSITY,
            "tile_size": TILE_SIZE, "use_poisson": USE_POISSON,
        }
    )

    # ── JIT-ted train step (n_max, k_max as static args → stable recompile) ──
    @partial(jax.jit, static_argnums=(4, 5))
    def train_step(params, opt_state, x, y, n_max: int, k_max: int, depths, masks):
        def loss_fn(p):
            out = model.apply({"params": p}, x, depths, n_max, k_max, y, masks)
            return out["loss"], out

        (loss, out), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        return loss, grads, out

    @jax.jit
    def apply_grads(params, opt_state, grads):
        updates, new_opt_state = tx.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state

    # ── JIT-ted eval step (fixed depth = MEAN_DEPTH) ──────────────────────────
    n_eval = max(0, MEAN_DEPTH - BPTT_DEPTH)
    k_eval = min(MEAN_DEPTH, BPTT_DEPTH)

    @jax.jit
    def eval_step(params, x, y, masks):
        depths = jnp.full((x.shape[0],), MEAN_DEPTH, dtype=jnp.int32)
        out    = model.apply({"params": params}, x, depths, n_eval, k_eval, y, masks)
        return out["loss"]

    def evaluate(params, masks, n_batches: int = 20) -> float:
        losses = []
        for _ in range(n_batches):
            xv, yv = get_batch(val_buf)
            losses.append(float(eval_step(params, xv, yv, masks)))
        return sum(losses) / len(losses)

    # ── Training loop ─────────────────────────────────────────────────────────
    t0 = time.time()
    step_times = []

    for step in range(TOTAL_STEPS):
        x, y = get_batch(train_buf)

        # Depth sampling — MUST be outside jit (host sync on .max())
        key, depth_key = jax.random.split(key)
        depths, n_max, k_max = sample_depth_plan(
            depth_key, BATCH_SIZE, MEAN_DEPTH, MAX_DEPTH, BPTT_DEPTH, USE_POISSON
        )

        # ── Forward + backward ───────────────────────────────────────────────
        ts = time.time()
        loss, grads, _out = train_step(
            params, opt_state, x, y, n_max, k_max, depths, current_masks
        )
        loss.block_until_ready()   # sync for accurate timing
        step_times.append(time.time() - ts)

        # ── CMS score accumulation (before optimizer update) ──────────────────
        if step >= PRUNE_START:
            # Walk grads pytree and accumulate for each tracked MLP weight
            g_params = grads   # grads has same structure as params
            for cms_key in list(cms_reg.keys()):
                parts    = cms_key.split("/")
                blk_name = parts[0]    # e.g. "core_0"
                fc_name  = parts[-1]   # e.g. "fc1_w"
                try:
                    grad_w = g_params[blk_name]["mlp"][fc_name]
                    cms_reg[cms_key] = accumulate_scores(cms_reg[cms_key], grad_w, TILE_SIZE)
                except (KeyError, TypeError):
                    pass   # param not present in this layer variant

        # ── Optimizer update ──────────────────────────────────────────────────
        params, opt_state = apply_grads(params, opt_state, grads)

        # ── Score normalisation ───────────────────────────────────────────────
        if step > 0 and step % SCORE_INTERVAL == 0 and step >= PRUNE_START:
            for k in cms_reg:
                cms_reg[k] = score_step(cms_reg[k])

        # ── Pruning ───────────────────────────────────────────────────────────
        if (PRUNE_START <= step < PRUNE_END
                and step > PRUNE_START
                and (step - PRUNE_START) % PRUNE_INTERVAL == 0):
            prune_round += 1
            for k in cms_reg:
                cms_reg[k] = prune_tiles(cms_reg[k], PRUNE_FRACTION)
            current_masks = cms_to_masks(cms_reg)
            # Re-zero optimizer momentum for dead tiles
            opt_state = zero_opt_state_for_dead_tiles(opt_state, params, cms_reg)
            avg_density = sum(get_density(s) for s in cms_reg.values()) / len(cms_reg)
            print(f"  🔪 PRUNE R{prune_round} @ step {step} | density={avg_density:.1%}")

        # ── Logging ───────────────────────────────────────────────────────────
        if step % LOG_INTERVAL == 0:
            loss_val   = float(loss)
            ppl        = math.exp(min(loss_val, 20.0))
            avg_density = sum(get_density(s) for s in cms_reg.values()) / max(len(cms_reg), 1)
            sps        = 1.0 / (sum(step_times[-LOG_INTERVAL:]) / len(step_times[-LOG_INTERVAL:]))
            phase      = phase_label(step, prune_round)
            eta_h      = (TOTAL_STEPS - step) / sps / 3600

            print(f"step {step:6d} [{phase:>12s}] | "
                  f"loss {loss_val:.4f} | ppl {ppl:8.2f} | "
                  f"density {avg_density:.2%} | "
                  f"{sps:.1f} step/s | eta {eta_h:.1f}h")

            wandb.log({
                "train/loss":     loss_val,
                "train/ppl":      ppl,
                "train/phase":    phase,
                "depth/n_max":    n_max,
                "depth/k_max":    k_max,
                "mlp/density":    avg_density,
                "perf/step_s":    1.0 / sps,
                "perf/steps_per_s": sps,
            }, step=step)

        # ── Evaluation ────────────────────────────────────────────────────────
        if step % EVAL_INTERVAL == 0 and step > 0:
            val_loss = evaluate(params, current_masks)
            val_ppl  = math.exp(min(val_loss, 20.0))
            print(f"  📊 EVAL @ step {step} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}")
            wandb.log({"val/loss": val_loss, "val/ppl": val_ppl}, step=step)

    # ── Final eval ────────────────────────────────────────────────────────────
    val_loss = evaluate(params, current_masks, n_batches=50)
    val_ppl  = math.exp(min(val_loss, 20.0))
    elapsed  = time.time() - t0
    print(f"\n{'='*70}")
    print(f"Training complete: {TOTAL_STEPS} steps in {elapsed/3600:.1f}h")
    print(f"Final val PPL: {val_ppl:.2f}  |  density: {FINAL_DENSITY:.1%}")
    print(f"{'='*70}")

    wandb.summary.update({
        "final_val_ppl":  val_ppl,
        "final_density":  FINAL_DENSITY,
        "n_params":        n_params,
        "effective_depth": EFFECTIVE_DEPTH,
    })
    wandb.finish()
    return params, cms_reg, current_masks


# ── Run training ──────────────────────────────────────────────────────────────
trained_params, cms_reg_final, final_masks = train()

## Cell 7 — Evaluation & Training Curve Plots

Fetch training history from wandb and plot:
1. **Loss & PPL vs step** — coloured by phase (DENSE / PRUNE Rn / SPARSE)
2. **MLP density over time**
3. **Depth stats** (n_max, k_max per step)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

# ─── Fetch run history from wandb ─────────────────────────────────────────────
api = wandb.Api()
runs = api.runs(
    f"{wandb.run.entity}/{WANDB_PROJECT}",
    filters={"display_name": EXPERIMENT_NAME}
)
assert runs, f"No wandb run found for '{EXPERIMENT_NAME}'"
run = runs[0]

history = run.history(
    keys=["train/loss", "train/ppl", "val/ppl", "mlp/density",
          "depth/n_max", "depth/k_max", "train/phase"],
    samples=TOTAL_STEPS,
)
df = pd.DataFrame(history).sort_values("_step")
print(f"Fetched {len(df)} log entries from wandb.")


# ─── Phase colour map ─────────────────────────────────────────────────────────
PHASE_COLORS = {
    "DENSE":  "#4C9BE8",
    "SPARSE": "#E8A04C",
}
# PRUNE phases get a gradient of reds
for r in range(1, N_PRUNE_ROUNDS + 1):
    frac = r / max(N_PRUNE_ROUNDS, 1)
    PHASE_COLORS[f"PRUNE R{r}"] = plt.cm.Reds(0.4 + 0.5 * frac)


def colour_by_phase(phases):
    return [PHASE_COLORS.get(str(p), "grey") for p in phases]


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(
    f"{EXPERIMENT_NAME}  |  d={D_MODEL}, T={MEAN_DEPTH}, final_density={FINAL_DENSITY:.0%}",
    fontsize=13, fontweight="bold"
)

# ── Plot 1: Train loss ────────────────────────────────────────────────────────
ax = axes[0]
colors = colour_by_phase(df["train/phase"].fillna("DENSE"))
ax.scatter(df["_step"], df["train/loss"], c=colors, s=2, alpha=0.6)
# Val PPL as secondary (smoothed)
df_val = df.dropna(subset=["val/ppl"])
if len(df_val) > 0:
    ax2 = ax.twinx()
    ax2.plot(df_val["_step"], df_val["val/ppl"], "k-o", markersize=4, lw=2, label="val PPL")
    ax2.set_ylabel("Val PPL", fontsize=10)
    ax2.legend(loc="upper right", fontsize=8)
ax.set_xlabel("Step")
ax.set_ylabel("Train Loss")
ax.set_title("Loss & Val PPL")
# Phase legend
patches = [mpatches.Patch(color=v, label=k) for k, v in PHASE_COLORS.items() if k in df["train/phase"].unique()]
ax.legend(handles=patches, loc="upper right", fontsize=7)

# ── Plot 2: MLP density ───────────────────────────────────────────────────────
ax = axes[1]
df_dens = df.dropna(subset=["mlp/density"])
ax.plot(df_dens["_step"], df_dens["mlp/density"] * 100, color="steelblue", lw=1.5)
ax.axhline(FINAL_DENSITY * 100, color="red", linestyle="--", lw=1, label=f"target {FINAL_DENSITY:.0%}")
ax.axvline(PRUNE_START, color="orange", linestyle=":", lw=1, label="prune start")
ax.axvline(PRUNE_END,   color="green",  linestyle=":", lw=1, label="prune end")
ax.set_xlabel("Step")
ax.set_ylabel("MLP Tile Density (%)")
ax.set_title("Pruning Schedule")
ax.legend(fontsize=8)
ax.set_ylim(0, 105)

# ── Plot 3: Depth stats ───────────────────────────────────────────────────────
ax = axes[2]
df_dep = df.dropna(subset=["depth/n_max", "depth/k_max"])
ax.plot(df_dep["_step"], df_dep["depth/n_max"] + df_dep["depth/k_max"], lw=1, label="total iters", color="navy")
ax.fill_between(df_dep["_step"],
                df_dep["depth/n_max"],
                df_dep["depth/n_max"] + df_dep["depth/k_max"],
                alpha=0.4, color="steelblue", label="k_max (grad)")
ax.fill_between(df_dep["_step"], 0, df_dep["depth/n_max"],
                alpha=0.3, color="grey", label="n_max (no-grad)")
ax.axhline(MEAN_DEPTH, color="red", linestyle="--", lw=1, label=f"mean_depth={MEAN_DEPTH}")
ax.set_xlabel("Step")
ax.set_ylabel("Iterations")
ax.set_title("Depth Sampling (Poisson)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{EXPERIMENT_NAME}_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {EXPERIMENT_NAME}_curves.png")

# ── Final stats summary ───────────────────────────────────────────────────────
final_val_ppl = df_val["val/ppl"].iloc[-1] if len(df_val) > 0 else float("nan")
print(f"\n{'─'*50}")
print(f"Experiment : {EXPERIMENT_NAME}")
print(f"Params     : {count_params(trained_params)/1e6:.1f}M")
print(f"d_model    : {D_MODEL}")
print(f"Depth T    : {MEAN_DEPTH}  (effective layers: {EFFECTIVE_DEPTH})")
print(f"Final dens : {FINAL_DENSITY:.1%}")
print(f"Val PPL    : {final_val_ppl:.2f}")
print(f"{'─'*50}")

## Cell 8 — Depth Scaling Analysis (Bonus)

After training, sweep over a range of fixed depths to plot the **depth–quality tradeoff curve**.
This tells you: for this trained model, how much depth do you actually need at inference?

Expected shape: quality improves quickly for small T, saturates around MEAN_DEPTH,
degrades slightly beyond (the injection gate was trained for a specific depth distribution).

Useful for deciding inference-time depth budgets.

In [ ]:
# ─── Evaluate at multiple fixed depths ────────────────────────────────────────

DEPTH_SWEEP = [1, 2, 3, 4, 6, 8, 10, 12, 16, 24]
N_EVAL_BATCHES = 30   # more batches = tighter confidence interval

depth_results = []   # list of (T, val_ppl)

for T in DEPTH_SWEEP:
    n_eval_t = max(0, T - BPTT_DEPTH)
    k_eval_t = min(T, BPTT_DEPTH)

    @partial(jax.jit, static_argnums=(3, 4))
    def eval_at_T(params, x, y, n_max, k_max, masks):
        depths = jnp.full((x.shape[0],), T, dtype=jnp.int32)
        out    = model.apply({"params": params}, x, depths, n_max, k_max, y, masks)
        return out["loss"]

    losses = []
    for _ in range(N_EVAL_BATCHES):
        xv, yv = get_batch(val_buf)
        l = eval_at_T(trained_params, xv, yv, n_eval_t, k_eval_t, final_masks)
        losses.append(float(l))

    avg_ppl = math.exp(min(sum(losses) / len(losses), 20.0))
    depth_results.append((T, avg_ppl))
    print(f"T={T:3d}: val PPL = {avg_ppl:.2f}")

# ─── Plot ─────────────────────────────────────────────────────────────────────
Ts, ppls = zip(*depth_results)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(Ts, ppls, "o-", color="steelblue", lw=2, markersize=7)
ax.axvline(MEAN_DEPTH, color="red", linestyle="--", lw=1.5, label=f"train mean_T={MEAN_DEPTH}")
ax.fill_betweenx([min(ppls) * 0.98, max(ppls) * 1.02],
                  BPTT_DEPTH, MEAN_DEPTH,
                  alpha=0.08, color="green", label=f"BPTT window ({BPTT_DEPTH}–{MEAN_DEPTH})")

# Annotate min PPL
min_T, min_ppl = min(depth_results, key=lambda x: x[1])
ax.annotate(f"  best: T={min_T}, PPL={min_ppl:.1f}",
            xy=(min_T, min_ppl), fontsize=10, color="darkgreen")

ax.set_xlabel("Fixed Depth T (inference)", fontsize=12)
ax.set_ylabel("Val PPL", fontsize=12)
ax.set_title(f"Depth–Quality Tradeoff | {EXPERIMENT_NAME}", fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{EXPERIMENT_NAME}_depth_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

# ─── Log to wandb ──────────────────────────────────────────────────────────────
depth_table = wandb.Table(columns=["depth_T", "val_ppl"], data=[[T, p] for T, p in depth_results])
wandb.log({"depth_sweep/table": depth_table,
           "depth_sweep/best_T": min_T,
           "depth_sweep/best_ppl": min_ppl})

print(f"\n📈 Depth sweep complete.")
print(f"Best inference depth: T={min_T} with PPL={min_ppl:.2f}")
print(f"Training mean_depth was: T={MEAN_DEPTH}")

## Cell 9 — Scaling Law Summary Table

If you run multiple experiments (different d_model / T / density), collect them here
from wandb and plot the iso-compute scaling law curves.

Each point: `(param_count, val_ppl)` coloured by the axis being swept.

In [ ]:
# ─── Pull all runs from the scaling project ────────────────────────────────────
api = wandb.Api()
all_runs = api.runs(f"{wandb.run.entity}/{WANDB_PROJECT}")

rows = []
for r in all_runs:
    cfg = r.config
    summ = r.summary
    val_ppl = summ.get("final_val_ppl") or summ.get("val/ppl")
    if val_ppl is None:
        continue
    rows.append({
        "name":           r.name,
        "d_model":        cfg.get("d_model",        "?"),
        "mean_depth":     cfg.get("mean_depth",     "?"),
        "effective_depth":cfg.get("effective_depth","?"),
        "final_density":  cfg.get("final_density",  1.0),
        "n_params":       cfg.get("n_params",        0),
        "val_ppl":        round(val_ppl, 2),
    })

if not rows:
    print("No completed runs found yet. Run more experiments and re-execute this cell.")
else:
    df_all = pd.DataFrame(rows).sort_values("val_ppl")
    print(df_all.to_string(index=False))

    # ── Scatter: params vs PPL, coloured by depth ─────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    sc = ax.scatter(
        df_all["n_params"] / 1e6,
        df_all["val_ppl"],
        c=df_all["mean_depth"].astype(float),
        cmap="viridis", s=80, zorder=3
    )
    plt.colorbar(sc, ax=ax, label="Mean Depth T")
    ax.set_xlabel("Parameters (M)")
    ax.set_ylabel("Val PPL")
    ax.set_title("Width Scaling (coloured by depth)")
    ax.grid(alpha=0.3)
    for _, row in df_all.iterrows():
        ax.annotate(f"  {row['name'][:12]}",
                    (row["n_params"] / 1e6, row["val_ppl"]), fontsize=7)

    ax = axes[1]
    sc2 = ax.scatter(
        df_all["effective_depth"].astype(float),
        df_all["val_ppl"],
        c=df_all["n_params"].astype(float) / 1e6,
        cmap="plasma", s=80, zorder=3
    )
    plt.colorbar(sc2, ax=ax, label="Params (M)")
    ax.set_xlabel("Effective Depth (prelude + core×T + coda)")
    ax.set_ylabel("Val PPL")
    ax.set_title("Depth Scaling (coloured by params)")
    ax.grid(alpha=0.3)

    plt.suptitle(f"Scaling Laws — {WANDB_PROJECT}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("scaling_laws.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Scaling law plot saved to scaling_laws.png")